# Code for inference with the MC-IDDPM

In [ ]:
import torch
from tqdm.auto import tqdm
import sys
sys.path.append("../..")
sys.path.append("..")
from train_mc_IDDPM import *
from MonaiDataLoader import MonaiDataLoader
from monai.inferers import SlidingWindowInferer

import os
import numpy as np
import SimpleITK as sitk
import nibabel as nib

from monai.transforms import (
    CropForeground
)
import torch.nn.functional as F

def pad_if_smaller_symmetric(in_tensor, patch_size):
    """
    Pads tensor symmetrically if D/H/W is smaller than the corresponding patch size.
    Assumes input tensor shape: (B, C, D, H, W)
    Returns padded tensor and the padding applied.
    """
    pad_d = max(patch_size[0] - in_tensor.shape[2], 0)
    pad_h = max(patch_size[1] - in_tensor.shape[3], 0)
    pad_w = max(patch_size[2] - in_tensor.shape[4], 0)

    pad_d1, pad_d2 = pad_d // 2, pad_d - pad_d // 2
    pad_h1, pad_h2 = pad_h // 2, pad_h - pad_h // 2
    pad_w1, pad_w2 = pad_w // 2, pad_w - pad_w // 2

    # F.pad expects (W_before, W_after, H_before, H_after, D_before, D_after)
    padding = (pad_w1, pad_w2, pad_h1, pad_h2, pad_d1, pad_d2)

    return F.pad(in_tensor, padding), padding

def unpad_if_smaller_symmetric(in_tensor, padding):
    pad_w1, pad_w2, pad_h1, pad_h2, pad_d1, pad_d2 = padding

    if pad_d1 + pad_d2 > 0:
        in_tensor = in_tensor[:, :, pad_d1:-pad_d2 if pad_d2 > 0 else None, :, :]
    if pad_h1 + pad_h2 > 0:
        in_tensor = in_tensor[:, :, :, pad_h1:-pad_h2 if pad_h2 > 0 else None, :]
    if pad_w1 + pad_w2 > 0:
        in_tensor = in_tensor[:, :, :, :, pad_w1:-pad_w2 if pad_w2 > 0 else None]
    return in_tensor

def invert_crop_foreground(mask, sampled_images, min_intensity):
    """
    Reverses the cropping operation by padding the sampled images back into their original positions 
    within the tensor shape defined by the mask's bounding box. Also converts back from RAS orientation to RAI.

    This function uses the bounding box computed from the mask to determine the region where the 
    sampled images should be placed. The rest of the tensor is filled with a default background value.

    Args:
        mask (torch.Tensor): A binary or multi-class mask tensor used to compute the bounding box 
                             for cropping. Shape: (B, C, D, H, W).
        sampled_images (torch.Tensor): The tensor containing the cropped predictions to be padded 
                                        back into the original tensor shape. Shape: (B, C, d, h, w).

    Returns:
        torch.Tensor: A tensor with the same shape as the mask, where the sampled images are placed 
                      back into their original positions, and the remaining regions are filled with 
                      a default background value.
    """
    # Initialize the CropForeground transform
    crop_foreground = CropForeground()

    # Compute the bounding box from the mask
    start, end = crop_foreground.compute_bounding_box(img=mask.numpy())
    #print(f"start: {start}")
    #print(f"end: {end}")
    #print(f"mask.numpy(): {mask.numpy().shape}")
    #print(f"sampled_images.shape: {sampled_images.shape}")

    # Create a tensor of zeros with the same shape as the mask
    padded = np.ones_like(mask.numpy())*min_intensity

    # Extract the bounding box coordinates
    x0, y0, z0 = start[1], start[2], start[3]
    x1, y1, z1 = end[1], end[2], end[3]

    # Place the sampled images back into the padded tensor
    padded[:, :, x0:x1, y0:y1, z0:z1] = sampled_images

    padded = np.flip(padded[0][0], axis=(0, 1)) 
    
    return padded

def save_tensor_as_mha(image_tensor, mask, in_mha_filename, out_mha_filename, max_value=2000, min_value=-1000):
    """
    Save a PyTorch tensor as a .mha file using metadata from another .mha file.

    Parameters:
        image_tensor (torch.Tensor): The image data to save.
        mask (torch.Tensor): The mask tensor used for cropping and padding.
        in_mha_filename (str): Path to the reference .mha file to copy metadata from.
        out_mha_filename (str): Path where the output .mha file will be saved.
        max_value (float): Max value for normalization.
        min_value (float): Min value for normalization.
    """
    # Read metadata from reference file
    original_image = sitk.ReadImage(in_mha_filename)
    original_array = sitk.GetArrayFromImage(original_image)

    # Squeeze & rescale to [min, max]
    data = image_tensor.detach().cpu().numpy()
    if data.ndim == 5:
        data = data[0,0]
    elif data.ndim == 4:
        data = data[0]
    
    if args.data_norm == 'ScaleIntensityRanged':
        # Rescale the values from (-1, 1) to (-1000, 2000) 
        # The predicted values should be able to 
        data = np.clip(data, -1.0, 1.0)
        data = (data + 1) / 2
        data = data * (max_value - min_value) + min_value
    elif args.data_norm == 'NormalizeIntensityd':
        normalization_stats_path = f"/projects/nian/synthrad2025/Dataset/{args.task}_Train_normalization_stats_{args.clip_min_ct}_{args.clip_max_ct}.json"
        with open(normalization_stats_path, "r") as stats_file:
            normalization_stats = json.load(stats_file)
        ct_mean = normalization_stats.get("ct_mean")
        ct_std = normalization_stats.get("ct_std")
        data = data * ct_std + ct_mean
    else:
        raise ValueError("'data_norm' should be either ScaleIntensityRanged or NormalizeIntensityd")

    # Pad back to the original shape
    data = invert_crop_foreground(
                mask=mask, 
                sampled_images=data,
                min_intensity=min_value)
    
    # Transpose input from (X, Y, Z) to (Z, Y, X)
    data = np.transpose(data, (2, 1, 0))

    # Check shape compatibility
    if data.shape != original_array.shape:
        raise ValueError(f"Shape mismatch: tensor shape {data.shape} != reference shape {data.shape}")
    
    # Convert to SimpleITK image
    new_image = sitk.GetImageFromArray(data)  # Ensure type consistency

    # Copy metadata (spacing, origin, direction)
    new_image.CopyInformation(original_image)

    # Write the image
    sitk.WriteImage(new_image, out_mha_filename)


#### Load model

In [ ]:
def load_model():
        A_to_B_model, filtered_dict, not_matching_keys = get_model(
                device=args.device,
                args=args
                )
        checkpoint = torch.load(join(args.resume), weights_only=False)
        A_to_B_model.load_state_dict(checkpoint['model_state_dict'])
        # Retrieve the epoch and best loss
        begin_epoch = checkpoint['epoch']
        best_loss = checkpoint['best_loss'] 
        print(f"Loaded from: {args.resume}")
        return A_to_B_model, begin_epoch, best_loss

#### Get data loader

In [ ]:
def get_dataloop():
        data_list_task_train, data_list_task_val = get_data_list(
                dataset_path=args.dataset_path, 
                task_datasplit_json=join(args.dataset_path, f"{args.task}_data_split.json"), # Assumes dataplit to be in the root folder of Dataset 
                task=args.task, 
                region=args.region,
                args=args
                )
        train_dataloader, val_dataloader, train_transforms, val_transforms = get_dataloader(
                data_list_task_train=data_list_task_train,
                data_list_task_val=data_list_task_val,
                task=args.task,
                args=args
                )
        return val_dataloader


### Run infer

In [ ]:
def predict_loop(val_dataloader, diffusion_sampling, A_to_B_model, exp_name, max_intensity, min_intensity):
    for i, batch in enumerate(val_dataloader):
        if True:#i==8:
            start_time = time.time() 
            with torch.no_grad():
                condition = batch['mri'].to(args.device) 
                target = batch['ct'].to(args.device) 

                condition, condition_padding = pad_if_smaller_symmetric(condition, args.patch_size)
                target, target_padding = pad_if_smaller_symmetric(target, args.patch_size)

                condition = unpad_if_smaller_symmetric(condition, condition_padding)
                target = unpad_if_smaller_symmetric(target, target_padding)

                mri_file_path = batch['mri_meta_dict']['filename_or_obj'][0]
                region = batch['mri_meta_dict']['filename_or_obj'][0].split('/')[-3]
                patient_id = batch['mri_meta_dict']['filename_or_obj'][0].split('/')[-2]
                
                # Prediction
                with torch.amp.autocast("cuda"):
                    sampled_images = inferer(
                    condition,
                    diffusion_sampling,
                    A_to_B_model
                    )

                sampled_images = unpad_if_smaller_symmetric(sampled_images, condition_padding)
                
                print(f"sampled_images.max(): {sampled_images.max()}")
                print(f"sampled_images.min(): {sampled_images.min()}")

                
                
                mha_filename = f'/projects/nian/synthrad2025/experiments/test/{exp_name}/{region}/{patient_id}/ct_pred.mha'
                os.makedirs(mha_filename.replace('/ct_pred.mha', ''), exist_ok=True)
                save_tensor_as_mha(
                    image_tensor=sampled_images,
                    mask=batch['mask'],
                    in_mha_filename=mri_file_path, 
                    out_mha_filename=mha_filename, 
                    max_value=max_intensity, 
                    min_value=min_intensity)
            print(f"File saved at {mha_filename}")
            print('Execution time:', '{:5.2f}'.format(time.time() - start_time), 'seconds')

## Regular DDIM

#### Define difussion process

In [ ]:
def get_diffusion(timestep_respacing, timestep_respacing_val, args):
    """
    Define the gaussian diffusion scheduler for training.
    These three parameters: training steps number, learning variance or not (using improved DDPM or original DDPM), and inference 
    timesteps number (only effective when using improved DDPM)
    In:
        timestep_respacing: Used mainly for inference to reduce the number of steps.
        timestep_respacing_val: Used mainly for inference to reduce the number of steps.
    Out:
        train_diffusion, val_diffusion, schedule_sampler: Diffusion models and the schedule sampler
        # val_diffusion has less time steps for inference
    """
    # Hard coded parameters
    #  
    sigma_small=False
    noise_schedule='linear'
    use_kl=False
    predict_xstart=True
    rescale_timesteps=True
    rescale_learned_sigmas=True 
    diffusion_steps=1000
    learn_sigma=True


    train_diffusion = create_gaussian_diffusion(
        steps=diffusion_steps,
        learn_sigma=learn_sigma,
        sigma_small=sigma_small,
        noise_schedule=noise_schedule,
        use_kl=use_kl,
        predict_xstart=predict_xstart,
        rescale_timesteps=rescale_timesteps,
        rescale_learned_sigmas=rescale_learned_sigmas,
        timestep_respacing=timestep_respacing,
    )

    val_diffusion = create_gaussian_diffusion(
        steps=diffusion_steps,
        learn_sigma=learn_sigma,
        sigma_small=sigma_small,
        noise_schedule=noise_schedule,
        use_kl=use_kl,
        predict_xstart=predict_xstart,
        rescale_timesteps=rescale_timesteps,
        rescale_learned_sigmas=rescale_learned_sigmas,
        timestep_respacing=timestep_respacing_val,
    )

    schedule_sampler = UniformSampler(train_diffusion)
    return train_diffusion, val_diffusion, schedule_sampler

In [ ]:
def diffusion_sampling(condition, model):
    sampled_images = val_diffusion.p_sample_loop(
        model,
        (condition.shape[0], 1, condition.shape[2], condition.shape[3],condition.shape[4]),
        condition=condition,
        clip_denoised=args.clip_denoised
        )
    # Check in sampled_images if there are any NaN values
    if torch.isnan(sampled_images).any():
        raise ValueError("Sampled images contain NaN values.")

    # /projects/nian/synthrad2025/trash
    affine = np.eye(4)
    nifti_img = nib.Nifti1Image(sampled_images.cpu().detach().numpy()[0][0], affine)
    nib.save(nifti_img, '/projects/nian/synthrad2025/trash/output_image.nii.gz')
    return sampled_images

#### Arguments

In [ ]:
#### Arguments
from argparse import Namespace

args = Namespace()
## To change
args.region_clip = False
args.clip_min_ct = -1024
args.clip_max_ct = 3000
args.clip_denoised = False
args.data_norm = "NormalizeIntensityd"
args.patch_size = (128, 128, 32)
args.sw_batch_size = 16
args.overlap = 0.5
args.overlap_mode = 'constant'
args.task = "Task1"
args.timestep_respacing_val = "50"
args.add_train_metric = []
if args.region_clip: 
    exp_name_L = ["MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_HN",
                "MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_TH",
                "MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB"]
    wandb_id_L = [
        "latest-run",
        "latest-run",
        "latest-run"]
    region_L = [
        ["HN"], 
        ["TH"],
        ["AB"]]
    max_intensity_L = [1700, 1400, 1400]
    min_intensity_L = [-1024, -1024 ,-1024]
    
else:
    #"MC-IDDPM_Task1_2_5000_timestep_1000_patchsize_2_SwinVIT_MSE_128_128_32_region_AB"
    exp_name = "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_128_128_32_NormalizeIntensityd_region_HN_TH_AB" 
    wandb_id = "run-20250509_185037-gnlx5pz2"#"latest-run"#"latest-run"
    args.region = ["HN", "TH", "AB"]
    max_intensity = 3000
    min_intensity = -1024
    
if args.task == "Task1":
    key_in = 'mr'
elif args.task == "Task2":
    key_in = 'cbct'

# Fairly constant
key_out = 'ct'
key_mask = 'mask'
args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
args.timestep_respacing = "50"
args.network = "SwinVIT"
args.dataset_path = "/projects/nian/synthrad2025/Dataset/"
args.cache_rate = 0
args.batch_size_train = 2
args.eval_metric = "L1"
args.train_metric = "MAE"
args.num_workers = 4
args.patch_num = 16
args.shuffle = False



In [ ]:
train_diffusion, val_diffusion, schedule_sampler = get_diffusion(
    timestep_respacing=args.timestep_respacing,
    timestep_respacing_val=args.timestep_respacing_val,
    args=args,
    )

inferer = SlidingWindowInferer(
    roi_size=(args.patch_size[0], args.patch_size[1], args.patch_size[2]), 
    sw_batch_size=args.sw_batch_size,
    overlap=args.overlap, 
    mode=args.overlap_mode, 
    progress=True
    )

if args.region_clip:
    print(f"Doing region_clip: {args.region_clip}") 
    for exp_idx, exp_name in enumerate(exp_name_L):
        wandb_id = wandb_id_L[exp_idx]
        args.region = region_L[exp_idx]
        max_intensity = max_intensity_L[exp_idx] 
        min_intensity = min_intensity_L[exp_idx] 
        args.path_checkpoint = f"/projects/nian/synthrad2025/results/MC-IDDPM/{exp_name}/wandb/{wandb_id}/files"
        args.resume = join(args.path_checkpoint, "model/A_to_B_model_latest.pt")

        print(f"Doing region: {args.region}") 

        val_dataloader = get_dataloop()
        A_to_B_model, begin_epoch, best_loss = load_model()

        if torch.cuda.device_count() > 1:
            print(f"Let's use {torch.cuda.device_count()} GPUs!")
            A_to_B_model = nn.DataParallel(A_to_B_model)
        A_to_B_model.to('cuda')
        A_to_B_model.eval()

        print('Epoch:', begin_epoch)
        print('best_loss:', best_loss)
        predict_loop(
            val_dataloader=val_dataloader, 
            diffusion_sampling=diffusion_sampling, 
            A_to_B_model=A_to_B_model, 
            exp_name=exp_name,
            max_intensity=max_intensity,
            min_intensity=min_intensity
        )
else:
    args.path_checkpoint = f"/projects/nian/synthrad2025/results/MC-IDDPM/{exp_name}/wandb/{wandb_id}/files"
    args.resume = join(args.path_checkpoint, "model/A_to_B_model_latest.pt")
    
    print(f"Doing region: {args.region}") 
    
    val_dataloader = get_dataloop()
    A_to_B_model, begin_epoch, best_loss = load_model()

    if torch.cuda.device_count() > 1:
        print(f"Let's use {torch.cuda.device_count()} GPUs!")
        A_to_B_model = nn.DataParallel(A_to_B_model)
    A_to_B_model.to('cuda')
    A_to_B_model.eval()

    print('Epoch:', begin_epoch)
    print('best_loss:', best_loss)
    predict_loop(
        val_dataloader=val_dataloader, 
        diffusion_sampling=diffusion_sampling, 
        A_to_B_model=A_to_B_model, 
        exp_name=exp_name,
        max_intensity=max_intensity,
        min_intensity=min_intensity
    )



## Using Diffusers DPMSolverMultistepScheduler (DPM++ 2M Karras)

In [ ]:
from diffusers import DPMSolverMultistepScheduler

In [ ]:
def get_diffusion(
    num_train_timesteps=1000,
    beta_start=0.0001,
    beta_end=0.02,
    beta_schedule='linear',
    prediction_type='sample',             # because predict_xstart=True
    variance_type='learned_range',        # because learn_sigma=True
    algorithm_type='dpmsolver++',         # high-quality sampling method
    solver_order=2,
    solver_type='midpoint',
    use_karras_sigmas=True,               # Optional for higher quality
    thresholding=True,                   # Optional
    dynamic_thresholding_ratio=0.995,     # Optional, only useful if thresholding=True
    sample_max_value=1.0,                 # Optional
    trained_betas=None,                   # Optional, used if you have custom betas
    num_inference_steps=50,               # Number of inference steps
    ):
    """
    Creates a DPM-Solver scheduler for diffusion models.

    Args:
        num_train_timesteps (int): Number of training timesteps.
        beta_start (float): Starting value of beta for the noise schedule.
        beta_end (float): Ending value of beta for the noise schedule.
        beta_schedule (str): Type of beta schedule ('linear', 'cosine', etc.).
        prediction_type (str): Type of prediction ('sample' or 'xstart').
        variance_type (str): Type of variance ('fixed', 'learned', 'learned_range').
        algorithm_type (str): Algorithm type for the solver ('dpmsolver', 'dpmsolver++', 'sde-dpmsolver' or 'sde-dpmsolver++').
        solver_order (int): Order of the solver (1, 2, or 3).
        solver_type (str): Type of solver ('midpoint', 'heun', etc.).
        use_karras_sigmas (bool): Whether to use Karras sigmas for improved quality.
        thresholding (bool): Whether to apply thresholding during sampling.
        dynamic_thresholding_ratio (float): Ratio for dynamic thresholding (if enabled).
        sample_max_value (float): Maximum value for sampling (if thresholding is enabled).
        trained_betas (list or None): Custom beta values (if provided).

    Returns:
        DPMSolverMultistepScheduler: Configured scheduler for diffusion sampling.
    """
    scheduler = DPMSolverMultistepScheduler(
        num_train_timesteps=num_train_timesteps,
        beta_start=beta_start,
        beta_end=beta_end,
        beta_schedule=beta_schedule,
        prediction_type=prediction_type,
        variance_type=variance_type,
        algorithm_type=algorithm_type,
        solver_order=solver_order,
        solver_type=solver_type,
        use_karras_sigmas=use_karras_sigmas,
        thresholding=thresholding,
        dynamic_thresholding_ratio=dynamic_thresholding_ratio,
        sample_max_value=sample_max_value,
        trained_betas=trained_betas,
    )
    scheduler.set_timesteps(num_inference_steps=num_inference_steps)
    scheduler.set_timesteps(num_inference_steps=num_inference_steps)
    return scheduler

In [ ]:
def diffusion_sampling(
    condition, 
    model):
    """
    Perform diffusion sampling using the DIFFUSERS library.

    Args:
    condition (torch.Tensor): The conditioning input tensor.
    model (torch.nn.Module): The model used for diffusion sampling.

    Returns:
    torch.Tensor: The final sampled tensor after the diffusion process.
    """
    scheduler = get_diffusion(
    use_karras_sigmas=args.use_karras_sigmas,
    prediction_type=args.prediction_type,
    variance_type=args.variance_type,
    num_inference_steps=args.num_inference_steps,
    algorithm_type=args.algorithm_type
    )

    final_scan = torch.randn_like(condition).cuda()
    model_kwargs = {}
    B, C = condition.shape[:2]
    with torch.no_grad():
        for timestep in tqdm(scheduler.timesteps, desc="Processing timesteps"):
            t = torch.tensor([timestep] * final_scan.shape[0]).cuda()
            # Predict denoised image for step t
            sample_in = scheduler.scale_model_input(final_scan, timestep)
            input_model = torch.cat([sample_in, condition], dim=1)
            model_output = model(input_model, t, **model_kwargs) 
            model_output_denoised, model_var_values = torch.split(model_output, C, dim=1)
            # Step the scheduler with both the denoised image and the variance
            out = scheduler.step(model_output=model_output_denoised, variance_noise=model_var_values, timestep=timestep, sample=final_scan)
            # Create input for next step
            final_scan = out.prev_sample
            
    return final_scan

In [ ]:
#### Arguments
from argparse import Namespace

args = Namespace()

args.use_karras_sigmas = True
args.algorithm_type = 'dpmsolver++'
args.prediction_type = 'sample'
args.variance_type = 'learned_range'
args.num_inference_steps = 50


## To change
args.region_clip = True
args.patch_size = (128, 128, 32)
args.sw_batch_size = 16
args.overlap = 0.5
args.task = "Task1"
args.clip_min_ct = -1000
args.clip_max_ct = 2000
args.data_norm = "ScaleIntensityRanged"

if args.region_clip: 
    exp_name_L = [None,
                None,
                "MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB"]
    wandb_id_L = [
        "latest-run",
        "latest-run",
        "latest-run"]
    region_L = [
        ["HN"], 
        ["TH"],
        ["AB"]]
    max_intensity_L = [1700, 1400, 1400]
    min_intensity_L = [-1024, -1024 ,-1024]
    
else:
    exp_name = "MC-IDDPM_Task1_2_5000_timestep_1000_patchsize_2_SwinVIT_MSE_128_128_32_region_AB"
    wandb_id = "latest-run"
    args.region = ["AB"]
    max_intensity = 2000
    min_intensity = -1000

if args.task == "Task1":
    key_in = 'mr'
elif args.task == "Task2":
    key_in = 'cbct'

# Fairly constant
key_out = 'ct'
key_mask = 'mask'
args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
args.timestep_respacing = "50"
args.network = "SwinVIT"
args.dataset_path = "/projects/nian/synthrad2025/Dataset/"
args.cache_rate = 0
args.batch_size_train = 2
args.eval_metric = "L1"
args.train_metric = "MAE"
args.num_workers = 4
args.patch_num = 16
args.shuffle = False

In [ ]:
inferer = SlidingWindowInferer(
        roi_size=(args.patch_size[0], args.patch_size[1], args.patch_size[2]), 
        sw_batch_size=args.sw_batch_size,
        overlap=args.overlap, 
        mode='gaussian', 
        progress=True
        )

if args.region_clip:
    print(f"Doing region_clip: {args.region_clip}") 
    for exp_idx, exp_name in enumerate(exp_name_L):
        if exp_name is None:
            continue
        wandb_id = wandb_id_L[exp_idx]
        args.region = region_L[exp_idx]
        max_intensity = max_intensity_L[exp_idx] 
        min_intensity = min_intensity_L[exp_idx] 
        args.path_checkpoint = f"/projects/nian/synthrad2025/results/MC-IDDPM/{exp_name}/wandb/{wandb_id}/files"
        args.resume = join(args.path_checkpoint, "model/A_to_B_model_latest.pt")

        print(f"Doing region: {args.region}") 

        val_dataloader = get_dataloop()
        A_to_B_model, begin_epoch, best_loss = load_model()
     

        if torch.cuda.device_count() > 1:
            print(f"Let's use {torch.cuda.device_count()} GPUs!")
            A_to_B_model = nn.DataParallel(A_to_B_model)
        A_to_B_model.to('cuda')
        A_to_B_model.eval()

        print('Epoch:', begin_epoch)
        print('best_loss:', best_loss)
        predict_loop(
            val_dataloader=val_dataloader, 
            diffusion_sampling=diffusion_sampling, 
            A_to_B_model=A_to_B_model, 
            exp_name=exp_name,
            max_intensity=max_intensity,
            min_intensity=min_intensity
        )
else:
    args.path_checkpoint = f"/projects/nian/synthrad2025/results/MC-IDDPM/{exp_name}/wandb/{wandb_id}/files"
    args.resume = join(args.path_checkpoint, "model/A_to_B_model_latest.pt")
    
    print(f"Doing region: {args.region}") 
    
    val_dataloader = get_dataloop()
    A_to_B_model, begin_epoch, best_loss = load_model()

    if torch.cuda.device_count() > 1:
        print(f"Let's use {torch.cuda.device_count()} GPUs!")
        A_to_B_model = nn.DataParallel(A_to_B_model)
    A_to_B_model.to('cuda')
    A_to_B_model.eval()

    print('Epoch:', begin_epoch)
    print('best_loss:', best_loss)
    predict_loop(
        val_dataloader=val_dataloader, 
        diffusion_sampling=diffusion_sampling, 
        A_to_B_model=A_to_B_model, 
        exp_name=exp_name,
        max_intensity=max_intensity,
        min_intensity=min_intensity
    )
